# 05 · Validation Plan — nsEM / SEC-MALS / native-MS + antigen-display extension

**Standard slot:** *validation plan.* **For Project 04 this means:** turn your ranked C3/C4/D2
candidate set into a controlled **experimental plan that determines the actual oligomeric state**
(in-silico filters only enrich), plus the **assembly design report** and the epitope-graft
antigen-display extension (vaccine framing). (D4/D5.)

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Select the ranked candidate set (the D★ deliverable)

Take the symmetry-clean survivors from notebook 03 and assemble the ranked C3/C4/D2 candidate
set — the core of the assembly design report. Each pick carries its symmetry, the three metrics,
and its wrong-oligomer status.

In [ ]:
import pandas as pd, os

pred = pd.read_csv("results/predictions.csv")
C = dict(scrmsd=2.5, plddt=80, pae=10, sym=2.0)
clean = pred[(pred["subunit_scrmsd"] <= C["scrmsd"]) & (pred["plddt"] >= C["plddt"]) &
             (pred["interface_pae"] <= C["pae"]) & (pred["symmetry_rmsd"] <= C["sym"]) &
             (pred["predicted_order"] == pred["n_subunits"])].copy()
clean["rank_score"] = (clean["plddt"]/100 + (10 - clean["interface_pae"])/10
                       + (2.5 - clean["subunit_scrmsd"]))
ranked = clean.sort_values("rank_score", ascending=False)
top_set = ranked.groupby("symmetry").head(3)   # top picks per symmetry
top_set.to_csv("results/candidate_set.csv", index=False)
print("ranked C3/C4/D2 candidate set (EXAMPLE_DATA on mock):", top_set.shape)
print(top_set[["assembly_id", "symmetry", "subunit_scrmsd", "interface_pae", "symmetry_rmsd"]].head(9))

## 2 · The experimental validation plan (oligomeric-state determination)

Generate the plan as a Markdown artifact. The key point: **interface pAE is necessary, not
sufficient** — the assays below are what actually determine the oligomeric state. Controls are
mandatory: positive = a known nanocage / a verified natural homo-oligomer of the target
symmetry; negative = a **scrambled-interface or monomeric** variant of your *own* design (must
NOT assemble); plus an unrelated-protein control.

In [ ]:
plan = """# Assembly Validation Plan (Project 04 — <your name>, <date>)

Goal: determine the ACTUAL oligomeric state of the ranked C3/C4/D2 candidates.
In-silico interface pAE is necessary, not sufficient — these assays confirm the real state.

## Expression & purification
- Host: E. coli BL21(DE3), 16-18 C overnight (mammalian only if a glycosylated antigen is grafted).
- Purify by IMAC -> SEC; carry the SEC trace into characterization.

## Oligomeric-state determination (the core)
| Assay | What it tells you | Go/No-go |
|-------|-------------------|----------|
| SEC | apparent size / homogeneity | single symmetric peak at expected volume |
| SEC-MALS | ABSOLUTE molar mass -> oligomeric number | mass = n_subunits x monomer mass |
| negative-stain EM (nsEM) | assembly architecture / particle shape | particles match target point group |
| native-MS | stoichiometry (intact assembly mass) | dominant species = intended order |
| (if warranted) cryo-EM | high-res structure of the assembly | matches the design |

## Controls (mandatory)
- Positive: a known nanocage / a VERIFIED natural homo-oligomer of the target symmetry.
- Negative: a scrambled-interface OR monomeric variant of the SAME design -> must NOT assemble.
- Unrelated-protein control.

## Honest reporting
- Report the assembly-success rate (N forming the intended state / N expressed), per symmetry.
- Wrong-oligomer outcomes are common; report them. NEVER claim a cage "will assemble."

## Timeline & cost (fill in)
- Weeks, reagents, instrument time (SEC-MALS / EM grid prep / native-MS) — cost it out.
"""
os.makedirs("results", exist_ok=True)
open("results/validation_plan.md", "w").write(plan)
print("wrote results/validation_plan.md — fill in the <...> fields and the costed timeline.")

## 3 · Antigen-display extension (epitope graft) `[extension]`

Nanocages are vaccine platforms: graft a **neutralizing or benign** antigen onto the symmetric
subunit so the particle displays many copies (the RSV-F / SARS-CoV-2-RBD nanoparticle-vaccine
paradigm). **Vaccine framing only** — see Responsible Research below. Sketch the construct and
the display-confirmation assay; do not model anything intended to enhance pathogen fitness.

In [ ]:
# Sketch the antigen-display construct (modelling lives on Colab with the real multimer backend).
antigen_display = {
    "scaffold": "top C3/C4/D2 candidate from results/candidate_set.csv",
    "antigen": "<a NEUTRALIZING / benign epitope, e.g. a stabilized viral fusion-protein antigen>",
    "linker": "short flexible or rigid fusion at the symmetry-exposed terminus",
    "copies_displayed": "= oligomeric order x antigens-per-subunit",
    "display_check": "bind a KNOWN neutralizing antibody to the particle (nsEM / BLI) to confirm display",
    "out_of_scope": "anything enhancing pathogen fitness/transmissibility/virulence; toxin display",
}
for k, v in antigen_display.items():
    print(f"  {k:18s}: {v}")
print("\nVaccine/neutralizing framing ONLY. See Responsible Research (README §, MANUAL §8).")

## 4 · (Stretch) larger symmetry — tetrahedral cages `[stretch]`

Higher point groups (tetrahedral / octahedral / icosahedral) make multi-subunit **cages**, not
just rings. They need a different protocol (often two-component) and far more compute (A100/HPC).
Note the path; treat as MSc-track future work unless you have the budget.

In [ ]:
from sym_tools import SYMMETRY_ORDER
print("higher point groups (order = # subunits):")
for s in ["T", "O", "I"]:
    print(f"  {s}: {SYMMETRY_ORDER[s]} subunits  (STRETCH — different protocol, A100/HPC; verify)")
print("\nFor a tetrahedral cage, plan a two-component design protocol and budget A100/HPC time.")

## D4 / D5 checklist
- [ ] `results/candidate_set.csv`: ranked C3/C4/D2 picks (symmetry-clean, predicted order = intended).
- [ ] `results/validation_plan.md`: SEC-MALS + nsEM + native-MS plan with controls (positive = known nanocage/natural homo-oligomer; negative = scrambled-interface/monomeric; unrelated).
- [ ] Wrong-oligomer risk carried into the report; assembly-success rate stated honestly (no 'will assemble').
- [ ] Antigen-display extension sketched with **neutralizing/vaccine framing** + display-confirmation assay.
- [ ] Assembly design report assembled; thesis chapter + 15-min talk + `v1.0` tagged release.

You're done — a rigorously filtered, honestly reported symmetric-assembly campaign.